# 人物スポットライト動画レンダラー（Colab実行用）

このノートブックは、Googleドライブの `マイドライブ/spotlight_reel/` フォルダに置いた
`project.json`（スマホ用エディタで書き出したもの）と動画から、`render.py` でMP4を作ります。

「ランタイム → すべてのセルを実行」で、上から順に自動的に処理されます。

1. リポジトリの取得・依存関係のインストール
2. Googleドライブをマウント
3. `project.json` を読み込み、動画を自動で探して `render.py` を実行
4. 成功した場合のみ、出力動画をノートブック内で再生し、スマホへダウンロード

途中で失敗した場合（ドライブ未接続・`project.json` が無い・動画が見つからない・
`render.py` のエラーなど）は、原因をこのノートブック上に大きく表示します。

## 1. リポジトリの取得・依存関係のインストール

In [ ]:
import os

REPO_DIR = "/content/spotlight-reel"

if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/digital-twin-creator/spotlight-reel.git {REPO_DIR}
else:
    print("リポジトリは既に取得済みです:", REPO_DIR)

%cd {REPO_DIR}
!pip install -q -r requirements.txt

# ffmpeg が無い場合のみ有効化してください（Colabには通常プリインストール済みです）
# !apt-get -y install ffmpeg

## 2. Googleドライブをマウント

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print("Googleドライブのマウントに失敗しました:", e)

## 3. project.json を読み込み、動画を自動で探して render.py を実行

- `project.json` は `/content/drive/MyDrive/spotlight_reel/project.json` に固定です。
- 動画は `project.json` 内の `"video"` の**ファイル名本体（拡張子を除いた部分）**で、
  同じフォルダの中から拡張子を問わず探します
  （例：`"video": "input.mov"` でも、フォルダ内に `input.mp4` があればそれを使います）。
  見つからない場合は、フォルダ内で**最終更新日時が最も新しい動画ファイル**を使います。
- 途中で失敗した場合（ドライブ未接続／`project.json` が無い／動画が見つからない／
  `render.py` のエラー）は、原因をこのセルの出力に大きく表示し、
  そこで処理を止めます（次のセルでのダウンロードは行われません）。

In [ ]:
import datetime
import html as _html
import json as _json
import subprocess

from IPython.display import HTML, Video, display

VIDEO_EXTS = (".mp4", ".mov", ".m4v", ".avi", ".mkv", ".webm")

DRIVE_ROOT = "/content/drive"
MYDRIVE = os.path.join(DRIVE_ROOT, "MyDrive")
DRIVE_DIR = os.path.join(MYDRIVE, "spotlight_reel")
JSON_PATH = os.path.join(DRIVE_DIR, "project.json")

OUT_PATH = None
SUCCESS = False


def show_error(title, detail=""):
    body = f'<div style="font-size:22px;font-weight:bold;color:#ff6b6b;">❌ {_html.escape(title)}</div>'
    if detail:
        body += (
            '<div style="font-size:14px;color:#ffd6d6;margin-top:8px;'
            'white-space:pre-wrap;font-family:monospace;">'
            + _html.escape(detail) + "</div>"
        )
    display(HTML(
        '<div style="background:#3a0f0f;border:2px solid #ff4d4f;border-radius:10px;'
        f'padding:16px 20px;margin:10px 0;">{body}</div>'
    ))


def show_success(title, detail=""):
    body = f'<div style="font-size:22px;font-weight:bold;color:#33cc88;">✅ {_html.escape(title)}</div>'
    if detail:
        body += (
            '<div style="font-size:14px;color:#d6ffe9;margin-top:8px;'
            'white-space:pre-wrap;font-family:monospace;">'
            + _html.escape(detail) + "</div>"
        )
    display(HTML(
        '<div style="background:#0f3a22;border:2px solid #33cc88;border-radius:10px;'
        f'padding:16px 20px;margin:10px 0;">{body}</div>'
    ))


if not os.path.isdir(MYDRIVE):
    show_error(
        "Googleドライブが接続されていません",
        "1つ上のセルで drive.mount() が正常に完了しているか確認してから、\n"
        "このセルをもう一度実行してください。"
    )
elif not os.path.isfile(JSON_PATH):
    show_error(
        "project.json が見つかりません",
        "スマホ用エディタで書き出した project.json を、次の場所に置いてから\n"
        "このセルをもう一度実行してください。\n\n" + JSON_PATH
    )
else:
    try:
        with open(JSON_PATH, "r", encoding="utf-8") as f:
            _project = _json.load(f)
    except Exception as e:
        show_error("project.json の読み込みに失敗しました", f"{type(e).__name__}: {e}")
        _project = None

    if _project is not None:
        video_name = _project.get("video") or ""
        stem = os.path.splitext(os.path.basename(video_name))[0] if video_name else ""

        VIDEO_PATH = None
        try:
            entries = sorted(os.listdir(DRIVE_DIR))
        except Exception as e:
            entries = []
            show_error("動画フォルダを読み込めませんでした", f"{DRIVE_DIR}\n{type(e).__name__}: {e}")

        if stem:
            for fname in entries:
                fstem, fext = os.path.splitext(fname)
                if fstem == stem and fext.lower() in VIDEO_EXTS:
                    VIDEO_PATH = os.path.join(DRIVE_DIR, fname)
                    break

        if not VIDEO_PATH and entries:
            # ファイル名本体で見つからない場合は、フォルダ内で最新の動画にフォールバック
            candidates = [
                os.path.join(DRIVE_DIR, f) for f in entries
                if os.path.splitext(f)[1].lower() in VIDEO_EXTS
            ]
            if candidates:
                VIDEO_PATH = max(candidates, key=os.path.getmtime)
                print("動画名(" + repr(stem) + ")では見つからなかったため、"
                      "フォルダ内で最新の動画を使います:", VIDEO_PATH)

        if not VIDEO_PATH:
            show_error(
                "動画ファイルが見つかりません",
                "project.json の \"video\": " + repr(video_name) + " に対応する動画も、\n"
                + DRIVE_DIR + " 内の動画ファイルも見つかりませんでした。\n"
                "動画をこのフォルダに置いてから、もう一度実行してください。"
            )
        else:
            timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M")
            OUT_PATH = os.path.join(DRIVE_DIR, f"output_{timestamp}.mp4")
            cmd = ["python", "render.py", JSON_PATH, "--video", VIDEO_PATH, "--out", OUT_PATH]
            print("動画:", VIDEO_PATH)
            print("JSON:", JSON_PATH)
            print("出力:", OUT_PATH)
            print("実行:", " ".join(cmd))
            result = subprocess.run(cmd, cwd=REPO_DIR, capture_output=True, text=True)
            print(result.stdout[-4000:])

            if result.returncode != 0 or not os.path.isfile(OUT_PATH):
                show_error(
                    "render.py の実行に失敗しました",
                    (result.stderr.strip()[-3000:] if result.stderr.strip() else "詳細不明のエラーです。標準出力を確認してください。")
                )
                OUT_PATH = None
            else:
                SUCCESS = True
                show_success("レンダリングが完了しました", OUT_PATH)

## 4. 出力の確認・スマホへのダウンロード
セル3が成功した場合のみ、ノートブック内で動画を再生し、続けて `files.download` で
スマホへのダウンロードを開始します（失敗している場合は何もしません）。

In [ ]:
if SUCCESS and OUT_PATH and os.path.isfile(OUT_PATH):
    print("出力ファイル:", OUT_PATH)
    print("サイズ:", os.path.getsize(OUT_PATH), "bytes")
    display(Video(OUT_PATH, embed=False, width=360))
    from google.colab import files
    files.download(OUT_PATH)
else:
    print("成功していないため、再生・ダウンロードはスキップしました。上の出力を確認してください。")